In [ ]:
SPINNER_HTML = '''
<div style="display:flex;align-items:center;gap:12px;background:#fff3cd;padding:12px 16px;border-radius:6px">
  <div style="width:22px;height:22px;border:3px solid #ffc107;border-top-color:transparent;
              border-radius:50%;animation:spin 0.8s linear infinite"></div>
  <span>{msg}</span>
</div>
<style>@keyframes spin {{ to {{ transform: rotate(360deg); }} }}</style>
'''

import io
import os
import json
import time
import zipfile
import sys
from datetime import datetime
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd

try:
    import requests
    import urllib3
    import ipywidgets as widgets
    from ipyaggrid import Grid
    from IPython.display import display, HTML, clear_output
    from urllib3.exceptions import InsecureRequestWarning
    urllib3.disable_warnings(InsecureRequestWarning)
except ImportError as e:
    print(f"⚠️  Import Error: {e}")
    raise

# Import utils
from pathlib import Path
notebook_dir = Path('.').resolve()
sys.path.insert(0, str(notebook_dir.parent))
from utils import (
    NOMADAPIClient,
    clean_text,
    normalize_value,
    normalize_sample_id,
    coerce_value,
    make_row_by_key,
    wait_for_sample_ids,
    resolve_name_to_reference,
    resolve_reference_to_name,
)

# Add custom CSS
display(HTML("""
    <style>
        /* General styling improvements */
        .jupyter-widgets {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        
        /* Button styling */
        .widget-button {
            background: linear-gradient(135deg, #274b8e 0%, #3b6db0 100%) !important;
            border: none !important;
            border-radius: 8px !important;
            color: white !important;
            font-weight: 600 !important;
            transition: all 0.3s ease !important;
            box-shadow: 0 4px 15px rgba(102, 126, 234, 0.3) !important;
        }
        
        .widget-button:hover {
            transform: translateY(-2px) !important;
            box-shadow: 0 8px 25px rgba(102, 126, 234, 0.4) !important;
        }
        
        /* Dropdown styling */
        .widget-dropdown select {
            border: 2px solid #e1e8ed !important;
            border-radius: 8px !important;
            padding: 8px 12px !important;
            font-size: 14px !important;
        }
        
        /* Output area styling */
        .jupyter-widgets-output-area {
            background: #f8f9fa;
            border-radius: 10px;
            padding: 15px;
            margin: 10px 0;
            border-left: 4px solid #28a745;
        }
        
        /* AG Grid custom styling */
        

    </style>"""))

In [ ]:
# API Configuration
URL_BASE = 'https://nomad08.csn29.bessy.de'
URL = f'{URL_BASE}/nomad-oasis/api/v1'
TOKEN = os.environ.get('NOMAD_CLIENT_ACCESS_TOKEN', '')
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept': 'application/json',
}

# Initialize API client
api_client = NOMADAPIClient(URL_BASE, HEADERS)

PROCESS_SETTLE_SECONDS = 2
PROCESS_RETRY_COUNT = 2

In [ ]:
# Schema package prefixes
PLUGIN_PREFIX = 'nomad_battery_space.schema_packages.hzb_bs_assembly_package'
BS_PACKAGE = 'nomad_battery_space.schema_packages.hzb_bs_package'

# Dashboard layout configuration
DASHBOARD_WIDTH = '2000px'

# Component schema definitions for reference resolution
ELECTRODE_SAMPLE_M_DEF = f'{BS_PACKAGE}.ElectrodeSample'
SEPARATOR_SAMPLE_M_DEF = f'{BS_PACKAGE}.SeparatorSample'
ELECTROLYTE_SAMPLE_M_DEF = f'{BS_PACKAGE}.ElectrolyteSample'

# Base schema definition for CoinCellBattery
BASE_SCHEMA = {
    'entry_type': 'CoinCell',
    'm_def': f'{PLUGIN_PREFIX}.CoinCellBattery',
    'required_keys': ['name', 'case_id'],
}

# Base columns (always visible)
BASE_COLUMNS = [
    ('name', 'Sample Name *', 'str'),
    ('case_id', 'Case ID *', 'str'),
    ('case_crimp', 'Case Crimp Method', 'str'),
    ('pressure_mpa', 'Pressure [MPa]', 'float'),
    ('working_electrode', 'Working Electrode (Reference)', 'str'),
    ('counter_electrode', 'Counter Electrode (Reference)', 'str'),
    ('reference_electrode', 'Reference Electrode (Ref)', 'str'),
    ('separator', 'Separator (Reference)', 'str'),
    ('electrolyte', 'Electrolyte (Reference)', 'str'),
    ('notes', 'Notes', 'str'),
]

# Initial schema (will be updated dynamically)
SCHEMAS = {
    'Coin Cell Battery': {**BASE_SCHEMA, 'columns': BASE_COLUMNS},
}

def get_columns_for_coincell():
    """Return column list for coin cell."""
    return BASE_COLUMNS.copy()

In [ ]:
def validate_row(schema_name, schema, row_by_key):
    """Validate a row against schema requirements."""
    missing = [key for key in schema['required_keys'] if not row_by_key.get(key)]
    if missing:
        missing_labels = []
        for key in missing:
            for column_key, column_label, _dtype in schema['columns']:
                if column_key == key:
                    missing_labels.append(column_label)
                    break
        return f"Missing required fields: {', '.join(missing_labels)}"
    return None

def extract_grid_frame():
    """Extract the current grid data as a DataFrame."""
    if grid is None:
        return pd.DataFrame(columns=col_labels)

    grid_data_out = getattr(grid, 'grid_data_out', {}) or {}
    candidate = grid_data_out.get('grid') if isinstance(grid_data_out, dict) else None

    if isinstance(candidate, pd.DataFrame):
        return candidate.copy().reset_index(drop=True)
    if isinstance(candidate, list):
        return pd.DataFrame(candidate).reset_index(drop=True)
    if isinstance(getattr(grid, 'grid_data', None), pd.DataFrame):
        return grid.grid_data.copy().reset_index(drop=True)
    return pd.DataFrame(columns=col_labels)

In [ ]:
def prepare_rows_from_frame(frame, schema_name, schema, existing_ids):
    """Prepare rows from grid data for upload."""
    prepared_rows = []
    row_errors = []

    for row_number, (_row_index, row) in enumerate(frame.iterrows(), start=1):
        row_by_key = make_row_by_key(row, col_keys, col_labels)
        if not any(value is not None for value in row_by_key.values()):
            continue

        validation_error = validate_row(schema_name, schema, row_by_key)
        if validation_error:
            sample_name = row_by_key.get('name') or f'row {row_number}'
            row_errors.append(f'{sample_name}: {validation_error}')
            continue

        sample_name = normalize_sample_id(row_by_key.get('name'))

        data = {
            'm_def': schema['m_def'],
            'name': sample_name,
            'datetime': date_picker.value.strftime('%Y-%m-%dT%H:%M:%S.%f'),
            'case_id': row_by_key.get('case_id'),
            'case_crimp': row_by_key.get('case_crimp', 'manual'),
            'substance_identifiers': {},
        }

        # Only add pressure if case_crimp is 'automatic'
        if row_by_key.get('case_crimp') == 'automatic' and row_by_key.get('pressure_mpa'):
            data['pressure'] = float(row_by_key['pressure_mpa'])

        # Handle component references with proper schema_m_def
        ref_fields_map = {
            'working_electrode': ELECTRODE_SAMPLE_M_DEF,
            'counter_electrode': ELECTRODE_SAMPLE_M_DEF,
            'reference_electrode': ELECTRODE_SAMPLE_M_DEF,
            'separator': SEPARATOR_SAMPLE_M_DEF,
            'electrolyte': ELECTROLYTE_SAMPLE_M_DEF,
        }
        
        for ref_field, schema_m_def in ref_fields_map.items():
            ref_input = row_by_key.get(ref_field)
            if ref_input:
                if '/archive/' in str(ref_input):
                    data[ref_field] = ref_input
                else:
                    ref = resolve_name_to_reference(api_client, ref_input, schema_m_def)
                    if ref:
                        data[ref_field] = ref
                    else:
                        row_errors.append(f"{sample_name}: Could not find {ref_field} named '{ref_input}'")

        if row_by_key.get('notes'):
            data['notes'] = {'description': row_by_key['notes']}

        file_name = f"{str(sample_name).replace(' ', '_')}.archive.json"
        prepared_rows.append({
            'row_number': row_number,
            'name': sample_name,
            'is_new': sample_name not in existing_ids,
            'file_name': file_name,
            'raw_path': file_name,
            'row_by_key': row_by_key,
            'archive': {'data': data},
        })

    return prepared_rows, row_errors

In [ ]:
# API wrapper functions (from utils)
def api_get(path, params=None):
    return requests.get(f'{api_client.url}{path}', headers=HEADERS, params=params, verify=False)

def api_post(path, json_body=None):
    return requests.post(f'{api_client.url}{path}', headers=HEADERS, json=json_body, verify=False)

def iter_archive_query(query, required=None, owner='visible', page_size=200):
    required = required or {'data': '*', 'metadata': '*'}
    page_after_value = None
    results = []

    while True:
        body = {
            'required': required,
            'owner': owner,
            'query': query,
            'pagination': {'page_size': page_size},
        }
        if page_after_value:
            body['pagination']['page_after_value'] = page_after_value

        response = api_post('/entries/archive/query', json_body=body)
        response.raise_for_status()
        payload = response.json()
        results.extend(payload.get('data', []))

        pagination = payload.get('pagination', {})
        page_after_value = pagination.get('next_page_after_value')
        if not page_after_value:
            break

    return results

def get_uploads():
    response = api_get('/uploads', params={'page_size': 200})
    response.raise_for_status()
    return response.json().get('data', [])

def get_upload_id(name):
    for upload in uploads:
        if upload.get('upload_name') == name:
            return upload.get('upload_id')
    return None

def matches_schema(entry, schema):
    archive = entry.get('archive', {})
    data = archive.get('data', {})
    metadata = archive.get('metadata', {})
    return (
        data.get('m_def') == schema['m_def']
        or metadata.get('entry_type') == schema['entry_type']
    )

def get_existing_samples(upload_id, schema):
    if not upload_id:
        return []

    all_entries = iter_archive_query({'upload_id': upload_id})
    filtered = [entry for entry in all_entries if matches_schema(entry, schema)]
    filtered.sort(key=lambda item: item.get('archive', {}).get('data', {}).get('name', ''))
    return filtered

In [ ]:
def render_existing_samples(upload_id, schema):
    """Return HTML widget for displaying existing coin cell samples."""
    existing = get_existing_samples(upload_id, schema) if upload_id else []
    
    if not existing:
        return HTML('<p style="color: #999; font-style: italic;">No existing coin cell samples in this upload</p>')
    
    # Build existing entries table
    existing_data = []
    for sample in existing:
        data = sample.get('archive', {}).get('data', {})
        notes = data.get('notes', {}) if isinstance(data.get('notes'), dict) else {}
        
        # Resolve component references to names
        we_ref = data.get('working_electrode')
        ce_ref = data.get('counter_electrode')
        re_ref = data.get('reference_electrode')
        sep_ref = data.get('separator')
        elect_ref = data.get('electrolyte')
        
        we_display = resolve_reference_to_name(api_client, we_ref)[1] if we_ref else ''
        ce_display = resolve_reference_to_name(api_client, ce_ref)[1] if ce_ref else ''
        re_display = resolve_reference_to_name(api_client, re_ref)[1] if re_ref else ''
        sep_display = resolve_reference_to_name(api_client, sep_ref)[1] if sep_ref else ''
        elect_display = resolve_reference_to_name(api_client, elect_ref)[1] if elect_ref else ''
        
        row = {
            'Lab ID': data.get('lab_id', ''),
            'Sample Name': data.get('name', 'N/A'),
            'Case ID': data.get('case_id', ''),
            'Case Crimp': data.get('case_crimp', ''),
            'Pressure [MPa]': data.get('pressure'),
            'Working Electrode': we_display,
            'Counter Electrode': ce_display,
            'Reference Electrode': re_display,
            'Separator': sep_display,
            'Electrolyte': elect_display,
            'Notes': notes.get('description', ''),
        }
        existing_data.append(row)
    
    df_existing = pd.DataFrame(existing_data)
    
    # Build HTML table manually with left-aligned text
    header_html = ''.join([
        f'<th style="text-align: left; padding: 8px; border-bottom: 2px solid #ccc; font-weight: 600; white-space: nowrap;">{col}</th>'
        for col in df_existing.columns
    ])
    
    rows_html = ''
    for _, row in df_existing.iterrows():
        cells = ''.join([
            f'<td style="text-align: left; padding: 6px 8px; border-bottom: 1px solid #eee;">{str(val) if val is not None else ""}</td>'
            for val in row
        ])
        rows_html += f'<tr>{cells}</tr>'
    
    html_table = f'''
    <div style="background: #f0f5ff; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 1px solid #ddd;">
        <h4 style="margin-top: 0; color: #333;">📋 Existing Coin Cell Samples in Upload</h4>
        <div style="overflow-x: auto; max-height: 300px; overflow-y: auto;">
            <table style="border-collapse: collapse; width: 100%; font-size: 0.9em; table-layout: auto;">
                <thead><tr>{header_html}</tr></thead>
                <tbody>{rows_html}</tbody>
            </table>
        </div>
    </div>
    '''
    return HTML(html_table)

In [ ]:
def show_grid():
    """Display existing samples (read-only) and new entry grid (editable)."""
    global grid, col_keys, col_labels
    
    schema_name = schema_dd.value
    schema = SCHEMAS[schema_name]
    upload_id = get_upload_id(upload_dd.value)
    
    # Update schema columns
    schema['columns'] = get_columns_for_coincell()
    
    col_keys = [column[0] for column in schema['columns']]
    col_labels = [column[1] for column in schema['columns']]
    
    # Display existing samples (read-only)
    out_existing.clear_output()
    with out_existing:
        widget_html = render_existing_samples(upload_id, schema)
        display(widget_html)
    
    # Query available component samples
    try:
        all_entries = iter_archive_query({})
        
        electrode_names = sorted(set([
            entry.get('archive', {}).get('data', {}).get('name')
            for entry in all_entries
            if (entry.get('archive', {}).get('data', {}).get('m_def') == ELECTRODE_SAMPLE_M_DEF
                and entry.get('archive', {}).get('data', {}).get('name'))
        ]))
        
        separator_names = sorted(set([
            entry.get('archive', {}).get('data', {}).get('name')
            for entry in all_entries
            if (entry.get('archive', {}).get('data', {}).get('m_def') == SEPARATOR_SAMPLE_M_DEF
                and entry.get('archive', {}).get('data', {}).get('name'))
        ]))
        
        electrolyte_names = sorted(set([
            entry.get('archive', {}).get('data', {}).get('name')
            for entry in all_entries
            if (entry.get('archive', {}).get('data', {}).get('m_def') == ELECTROLYTE_SAMPLE_M_DEF
                and entry.get('archive', {}).get('data', {}).get('name'))
        ]))
    except Exception as e:
        print(f"⚠️  Could not load component samples: {e}")
        electrode_names = []
        separator_names = []
        electrolyte_names = []
    
    # Create new entry grid
    df = pd.DataFrame(columns=col_labels)
    
    blank_rows = 12
    for _index in range(blank_rows):
        df.loc[len(df)] = pd.Series(dtype='object')
    
    # Build column definitions with select editors
    column_defs = []
    for label in df.columns:
        col_def = {'headerName': label, 'field': label}
        
        # Add dropdown for case_crimp
        if label == 'Case Crimp Method':
            col_def['cellEditor'] = 'agSelectCellEditor'
            col_def['cellEditorParams'] = {
                'values': ['', 'manual', 'automatic']
            }
        
        # Add dropdowns for component references
        elif label == 'Working Electrode (Reference)':
            col_def['cellEditor'] = 'agSelectCellEditor'
            col_def['cellEditorParams'] = {
                'values': [''] + electrode_names
            }
        
        elif label == 'Counter Electrode (Reference)':
            col_def['cellEditor'] = 'agSelectCellEditor'
            col_def['cellEditorParams'] = {
                'values': [''] + electrode_names
            }
        
        elif label == 'Reference Electrode (Ref)':
            col_def['cellEditor'] = 'agSelectCellEditor'
            col_def['cellEditorParams'] = {
                'values': [''] + electrode_names
            }
        
        elif label == 'Separator (Reference)':
            col_def['cellEditor'] = 'agSelectCellEditor'
            col_def['cellEditorParams'] = {
                'values': [''] + separator_names
            }
        
        elif label == 'Electrolyte (Reference)':
            col_def['cellEditor'] = 'agSelectCellEditor'
            col_def['cellEditorParams'] = {
                'values': [''] + electrolyte_names
            }
        
        column_defs.append(col_def)
    
    grid_options = {
        'columnDefs': column_defs,
        'defaultColDef': {'editable': True, 'resizable': True},
        'rowSelection': 'multiple',
        'enableRangeSelection': True,
        'stopEditingWhenCellsLoseFocus': True,
    }
    grid = Grid(
        grid_data=df,
        grid_options=grid_options,
        sync_on_edit=True,
        theme='ag-theme-balham',
        columns_fit='auto',
        index=False,
    )
    
    out_grid.clear_output()
    with out_grid:
        display(grid)

In [ ]:
def on_create(_button):
    """Handle click on Create button."""
    with out_status:
        clear_output(wait=True)
        schema_name = schema_dd.value
        schema = SCHEMAS[schema_name]
        upload_name = upload_dd.value
        upload_id = get_upload_id(upload_name)
        
        if not upload_id:
            display(render_status('error', '❌ No upload selected'))
            return
        
        # Extract grid data
        grid_data = extract_grid_frame()
        if grid_data.empty:
            display(render_status('error', '❌ No data in grid'))
            return
        
        # Get existing samples
        existing_before = get_existing_samples(upload_id, schema)
        existing_ids = {
            normalize_sample_id(entry.get('archive', {}).get('data', {}).get('name'))
            for entry in existing_before
        }
        
        # Prepare rows
        prepared_rows, row_errors = prepare_rows_from_frame(
            grid_data, schema_name, schema, existing_ids
        )
        
        if not prepared_rows:
            if row_errors:
                display(render_status('error', f'❌ {"<br/>".join(row_errors)}'))
            else:
                display(render_status('warning', '⚠️  No data rows to submit'))
            return
        
        if row_errors:
            display(render_status('warning', f'⚠️  Some rows had errors and were skipped:<br/>{"<br/>".join(row_errors)}'))
        
        # Track messages to re-display after clearing spinners
        messages = []
        
        # Upload
        display(HTML(SPINNER_HTML.format(msg='Uploading archive data...')))
        status, detail = api_client.write_archive_bundle_via_api(upload_id, prepared_rows)
        clear_output(wait=True)
        
        if status != 200:
            display(render_status('error', f'❌ Upload failed: {status}<br/>{detail}'))
            return
        
        upload_msg = render_status('success', f'✅ Uploaded {len(prepared_rows)} samples')
        display(upload_msg)
        messages.append(upload_msg)
        
        # Process upload
        display(HTML(SPINNER_HTML.format(msg='Processing upload...')))
        last_payload = None
        process_state_info = ''
        try:
            last_payload = api_client.process_upload(upload_id, timeout=180)
            
            # Display payload info for debugging
            payload_keys = list(last_payload.keys()) if last_payload else []
            payload_json = json.dumps(last_payload, indent=2, default=str)
            debug_html = f'''
            <div style="background: #f0f0f0; padding: 10px; border-radius: 6px; margin-bottom: 6px; font-family: monospace; font-size: 0.85em;">
                <strong>🔍 DEBUG - process_upload() response:</strong><br/>
                <strong>Keys:</strong> {payload_keys}<br/>
                <strong>Full payload:</strong><br/>
                <pre style="background: #fff; padding: 8px; border-radius: 4px; overflow-x: auto; max-height: 150px;">{payload_json}</pre>
            </div>
            '''
            display(HTML(debug_html))
            
            # Try different possible state keys
            state_value = (
                last_payload.get('state') 
                or last_payload.get('current_state') 
                or last_payload.get('processing_state')
                or last_payload.get('upload_state')
                or last_payload.get('status')
                or 'completed'
            )
            process_state_info = f'<br/>State: {state_value}'
            process_msg = render_status('success', f'✅ Upload processed successfully!{process_state_info}')
            display_error = False
        except TimeoutError as e:
            process_msg = render_status('warning', f'⚠️  {e}')
            display_error = True
        except Exception as e:
            process_msg = render_status('error', f'❌ Processing failed: {e}')
            display_error = True
        
        clear_output(wait=True)
        
        # Re-display upload message and new process message
        for msg in messages:
            display(msg)
        display(process_msg)
        
        if display_error and 'Processing failed' in str(process_msg.data):
            return
        
        messages.append(process_msg)
        
        # Wait for samples to appear
        display(HTML(SPINNER_HTML.format(msg='Waiting for samples to register...')))
        time.sleep(PROCESS_SETTLE_SECONDS)
        new_sample_ids = [row['name'] for row in prepared_rows]
        
        try:
            wait_for_sample_ids(api_client, upload_id, schema, new_sample_ids, timeout=60)
            wait_msg = render_status('success', '✅ All samples registered and processed!')
        except TimeoutError:
            wait_msg = render_status('warning', '⚠️  Samples may still be processing, please refresh')
        
        clear_output(wait=True)
        
        # Re-display all messages
        for msg in messages:
            display(msg)
        display(wait_msg)
        
        # Refresh grid
        time.sleep(1)
        show_grid()

def render_status(kind, message):
    colors = {
        'error': '#f8d7da',
        'warning': '#fff3cd',
        'success': '#d4edda',
        'info': '#d1ecf1',
    }
    return HTML(
        f'<div style="background:{colors[kind]};padding:10px;border-radius:6px;margin-bottom:6px">{message}</div>'
    )


In [ ]:
# Load uploads
uploads = get_uploads()
grid = None
col_keys = []
col_labels = []

upload_dd = widgets.Dropdown(
    options=[u.get('upload_name') for u in uploads if u.get('upload_name')],
    value=None,
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
)
schema_dd = widgets.Dropdown(
    options=list(SCHEMAS.keys()),
    value='Coin Cell Battery',
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
    disabled=True,
)
date_picker = widgets.NaiveDatetimePicker(
    value=datetime.now(), 
    disabled=False,
    style={'description_width': 'initial', 'font_size': '16px'}
)

out_grid = widgets.Output()
out_existing = widgets.Output()
out_status = widgets.Output()

In [ ]:
def on_context_change(_change):
    out_status.clear_output()
    show_grid()

upload_dd.observe(on_context_change, names='value')
schema_dd.observe(on_context_change, names='value')

btn_create = widgets.Button(
    description='✅ Upload & Process',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px'),
    tooltip='Create coin cell samples from the data grid and trigger NOMAD processing',
)
btn_refresh = widgets.Button(
    description='🔄 Refresh',
    layout=widgets.Layout(width='110px', height='40px'),
    tooltip='Reload existing entries from NOMAD',
)

btn_create.on_click(on_create)
btn_refresh.on_click(lambda _: show_grid())

# Build the dashboard layout
init_message = widgets.HTML(
    '<h3 style="background: #e3f2fd; padding: 12px 16px; border-radius: 8px; border-left: 4px solid #1976d2; color: #0d47a1;">'
    'ℹ️ Please select an upload to begin'
    '</h3>'
)

controls = widgets.VBox([
    widgets.HTML('<h3>Configuration</h3>'),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Type:</div>'),
        schema_dd
    ]),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">NOMAD Upload:</div>'),
        upload_dd
    ]),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Timestamp:</div>'),
        date_picker
    ]),
])

new_samples_message = widgets.HTML(
    '<h3 style="background: #e3f2fd; color: #0d47a1; padding: 12px 16px; border-radius: 8px; border-left: 4px solid #1976d2; margin: 15px 0 8px; font-size: 1.3em;">📝 Register new Coin Cell Samples &nbsp;<small style="color:#0d47a1">- edit cells directly, add rows at bottom</small></h3>'
)

buttons = widgets.HBox([btn_create, widgets.HTML('&nbsp;&nbsp;'), btn_refresh])

dashboard_banner = widgets.HTML(
    f'<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 20px; border-radius: 10px; color: white; margin-bottom: 10px; width: {DASHBOARD_WIDTH}; box-sizing: border-box;">'
    '<div style="margin:0; font-family: monospace; font-size: 2.2em;">🔋 Coin Cell Sample Batch Registration</div>'
    '<p style="margin:4px 0 0; opacity:0.6; font-size:1rem;">Voila Dashboard for batch registration of Coin Cell Battery entries in NOMAD</p>'
    '</div>'
)

dashboard_content = widgets.VBox([
    init_message,
    controls,
    out_existing,
    new_samples_message,
    out_grid,
    widgets.HTML('<div style="margin:8px 0; padding:10px; background:#fff3cd; border-radius:4px; border-left:4px solid #ffc107; color:#856404;">'
                 '<strong>⚠️ After editing:</strong> Click on an empty cell in the grid before pressing "Upload & Process"'
                 '</div>'),
    buttons,
    out_status,
])

# Display the complete dashboard with max-width container
display(dashboard_banner)
display(widgets.VBox([
            dashboard_content,
        ], 
        layout=widgets.Layout(width=DASHBOARD_WIDTH, margin='0')))